# NCF India Zenodo Recon

**Goal:** Understand the shape and processability of the `ncfindia` Zenodo community *before* building a pipeline.

Questions we want to answer:
1. How many records exist, broken down by resource type?
2. Who are the unique researchers / contributors?
3. Which records have the anchor file types we need: **CSV** (data), **README/TXT/PDF** (codebook), **SHP** (spatial)?
4. How many records are "pipeline-ready" (have at least a CSV + a codebook)?
5. What's the estimated storage footprint of the processable records?

This is pure metadata exploration — **no files are downloaded**.

In [ ]:
import requests
import json
import time
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

COMMUNITY = "ncfindia"
ZENODO_API = "https://zenodo.org/api/records"
CACHE_FILE = Path("../ncfindia_all_records.json")

print(f"Targeting community: {COMMUNITY}")

## 1. Fetch all record metadata

We page through the Zenodo API and collect every record in the community.
Results are cached to `../ncfindia_all_records.json` so subsequent runs are instant.

In [ ]:
def fetch_all_records(community, page_size=25):  # Zenodo API max page size is 25
    """Page through Zenodo API and return all records for a community."""
    records = []
    page = 1
    while True:
        params = {"communities": community, "size": page_size, "page": page}
        resp = requests.get(ZENODO_API, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        hits = data["hits"]["hits"]
        if not hits:
            break
        records.extend(hits)
        total = data["hits"]["total"]
        print(f"  Page {page}: fetched {len(hits)} records (total reported: {total})")
        if len(records) >= total:
            break
        page += 1
        time.sleep(0.3)  # be polite
    return records


if CACHE_FILE.exists():
    print("Loading from cache...")
    records = json.loads(CACHE_FILE.read_text())
else:
    print("Fetching from Zenodo API...")
    records = fetch_all_records(COMMUNITY)
    CACHE_FILE.write_text(json.dumps(records, indent=2))
    print(f"Cached {len(records)} records to {CACHE_FILE}")

print(f"\nTotal records: {len(records)}")

## 2. Build a flat DataFrame

Extract the fields we care about from each record's metadata and file list.

In [ ]:
def classify_file(filename):
    """Return a coarse anchor type for a filename."""
    fn = filename.lower()
    if fn.endswith(".csv"):
        return "csv"
    if fn.endswith(".shp"):
        return "shp"
    if fn.endswith(".pdf"):
        return "pdf"
    if fn.endswith(".txt") or fn.endswith(".md") or "readme" in fn:
        return "txt_readme"
    if fn.endswith(".xlsx") or fn.endswith(".xls"):
        return "excel"
    if fn.endswith(".r") or fn.endswith(".py") or fn.endswith(".ipynb"):
        return "code"
    if any(fn.endswith(ext) for ext in [".jpg", ".jpeg", ".png", ".tif", ".tiff"]):
        return "image"
    if fn.endswith(".zip") or fn.endswith(".gz") or fn.endswith(".tar"):
        return "archive"
    if fn.endswith(".geojson") or fn.endswith(".kml") or fn.endswith(".gpkg"):
        return "geo_other"
    return "other"


rows = []
for rec in records:
    meta = rec["metadata"]
    resource_type = meta.get("resource_type", {}).get("type", "unknown")
    creators = [c["name"] for c in meta.get("creators", [])]
    pub_date = meta.get("publication_date", "")

    files = rec.get("files", [])
    file_types = Counter(classify_file(f.get("filename") or f.get("key", "")) for f in files)
    total_bytes = sum(f.get("filesize", f.get("size", 0)) for f in files)

    rows.append({
        "id": rec["id"],
        "title": meta.get("title", "")[:80],
        "resource_type": resource_type,
        "pub_date": pub_date,
        "creators": "; ".join(creators),
        "n_creators": len(creators),
        "n_files": len(files),
        "total_mb": round(total_bytes / 1e6, 2),
        # anchor type counts
        "n_csv": file_types.get("csv", 0),
        "n_shp": file_types.get("shp", 0),
        "n_pdf": file_types.get("pdf", 0),
        "n_txt_readme": file_types.get("txt_readme", 0),
        "n_excel": file_types.get("excel", 0),
        "n_image": file_types.get("image", 0),
        "n_code": file_types.get("code", 0),
        "n_archive": file_types.get("archive", 0),
        "n_geo_other": file_types.get("geo_other", 0),
        "n_other": file_types.get("other", 0),
    })

df = pd.DataFrame(rows)
# pipeline-ready: has at least 1 CSV AND at least 1 codebook (txt/readme or pdf)
df["has_csv"] = df["n_csv"] > 0
df["has_codebook"] = (df["n_txt_readme"] > 0) | (df["n_pdf"] > 0)
df["has_shp"] = df["n_shp"] > 0
df["pipeline_ready"] = df["has_csv"] & df["has_codebook"]

print(df.shape)
df.head()

## 3. High-level counts

Total records by resource type, and the pipeline-readiness breakdown.

In [ ]:
print("=== Resource type breakdown ===")
print(df["resource_type"].value_counts().to_string())
print()

datasets = df[df["resource_type"] == "dataset"]
print(f"=== Dataset records only: {len(datasets)} ===")
print(f"  with CSV:           {datasets['has_csv'].sum()}")
print(f"  with codebook:      {datasets['has_codebook'].sum()}")
print(f"  with SHP:           {datasets['has_shp'].sum()}")
print(f"  pipeline-ready:     {datasets['pipeline_ready'].sum()}  (CSV + codebook)")
print()
print(f"=== Storage (pipeline-ready datasets) ===")
ready = datasets[datasets["pipeline_ready"]]
print(f"  Total size (all files incl. images): {ready['total_mb'].sum():.1f} MB")
# estimate: CSV + txt only
csv_txt_mb = ready.apply(
    lambda r: r["total_mb"] * (r["n_csv"] + r["n_txt_readme"] + r["n_pdf"]) / max(r["n_files"], 1),
    axis=1
)
print(f"  Estimated CSV+codebook only:          {csv_txt_mb.sum():.1f} MB (rough fraction estimate)")

## 4. Unique researchers

Who contributes to this community, and how many datasets does each person appear in?

In [ ]:
from collections import Counter

researcher_counts = Counter()
for _, row in datasets.iterrows():
    for name in row["creators"].split("; "):
        name = name.strip()
        if name:
            researcher_counts[name] += 1

print(f"Unique researchers across all datasets: {len(researcher_counts)}\n")
print("Top 20 by dataset count:")
for name, count in researcher_counts.most_common(20):
    print(f"  {count:3d}  {name}")

## 5. Researcher: Raman, T. R. Shankar

Zoom in on the author named in the design doc — how many of their records are pipeline-ready?

In [ ]:
AUTHOR = "Raman, T. R. Shankar"

raman = datasets[datasets["creators"].str.contains(AUTHOR, na=False)]
print(f"Records by {AUTHOR}: {len(raman)}")
print(f"  pipeline-ready: {raman['pipeline_ready'].sum()}")
print(f"  with SHP:       {raman['has_shp'].sum()}")
print(f"  total size:     {raman['total_mb'].sum():.1f} MB")
print()
print(raman[["id", "title", "pub_date", "n_csv", "n_txt_readme", "n_pdf", "n_shp", "n_image", "total_mb", "pipeline_ready"]].to_string(index=False))

## 6. Per-record file type breakdown (datasets only)

A table showing each dataset record and its anchor file counts — good for spotting patterns.

In [ ]:
cols = ["id", "title", "pub_date", "n_csv", "n_txt_readme", "n_pdf", "n_shp",
        "n_excel", "n_image", "n_code", "total_mb", "pipeline_ready"]
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 50)
datasets[cols].sort_values("pipeline_ready", ascending=False)

## 7. Visualisations

### 7a. Resource type distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- 7a: resource type pie ---
ax = axes[0]
type_counts = df["resource_type"].value_counts()
ax.pie(type_counts.values, labels=type_counts.index, autopct="%1.0f%%", startangle=90)
ax.set_title("All records by resource type")

# --- 7b: pipeline readiness bar ---
ax = axes[1]
labels = ["Total datasets", "Has CSV", "Has codebook", "Has SHP", "Pipeline-ready"]
values = [
    len(datasets),
    datasets["has_csv"].sum(),
    datasets["has_codebook"].sum(),
    datasets["has_shp"].sum(),
    datasets["pipeline_ready"].sum(),
]
bars = ax.bar(labels, values, color=["steelblue", "#4caf50", "#ff9800", "#9c27b0", "#f44336"])
ax.bar_label(bars, padding=2)
ax.set_title("Dataset pipeline-readiness funnel")
ax.set_ylabel("# records")
ax.tick_params(axis="x", rotation=30)
ax.set_ylim(0, max(values) * 1.2)

# --- 7c: top researchers ---
ax = axes[2]
top_n = 10
top_researchers = researcher_counts.most_common(top_n)
names = [n for n, _ in top_researchers]
counts = [c for _, c in top_researchers]
# shorten names: Last, First-initial
short_names = [n.split(",")[0] + ((", " + n.split(",")[1].strip()[0] + ".") if "," in n and len(n.split(",")) > 1 else "") for n in names]
bars = ax.barh(short_names[::-1], counts[::-1], color="teal")
ax.bar_label(bars, padding=2)
ax.set_title(f"Top {top_n} researchers (datasets)")
ax.set_xlabel("# datasets")

plt.tight_layout()
plt.savefig("../ncfindia_overview.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved ncfindia_overview.png")

### 7b. File-type breakdown per dataset (heatmap)

Rows = datasets, columns = anchor file types. Color = count of that file type.

In [ ]:
import numpy as np

file_cols = ["n_csv", "n_txt_readme", "n_pdf", "n_shp", "n_excel", "n_code", "n_image", "n_archive", "n_other"]
col_labels = [c.replace("n_", "") for c in file_cols]

# sort by pipeline-ready first, then by total files
plot_df = datasets.sort_values(["pipeline_ready", "n_files"], ascending=[False, False]).reset_index(drop=True)
matrix = plot_df[file_cols].values.astype(float)
# cap images at 5 for readability
img_idx = file_cols.index("n_image")
matrix[:, img_idx] = np.clip(matrix[:, img_idx], 0, 5)

row_labels = [f"{row['id']} — {row['title'][:40]}" for _, row in plot_df.iterrows()]

fig, ax = plt.subplots(figsize=(12, max(6, len(row_labels) * 0.35)))
im = ax.imshow(matrix, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=7)
plt.colorbar(im, ax=ax, label="file count (images capped at 5)")
ax.set_title("File-type breakdown per dataset (sorted: pipeline-ready first)")

# mark pipeline-ready boundary
n_ready = plot_df["pipeline_ready"].sum()
if n_ready > 0 and n_ready < len(plot_df):
    ax.axhline(n_ready - 0.5, color="blue", linewidth=1.5, linestyle="--", label="pipeline-ready / not-ready boundary")
    ax.legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.savefig("../ncfindia_file_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved ncfindia_file_heatmap.png")

### 7c. Storage breakdown for pipeline-ready records

In [ ]:
ready = datasets[datasets["pipeline_ready"]].copy()
ready = ready.sort_values("total_mb", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, len(ready) * 0.4)))
short_titles = [f"{row['id']}: {row['title'][:45]}" for _, row in ready.iterrows()]
bars = ax.barh(short_titles, ready["total_mb"], color="steelblue", label="Total (incl. images)")

# overlay csv-only estimate
csv_only = ready.apply(
    lambda r: r["total_mb"] * r["n_csv"] / max(r["n_files"], 1), axis=1
)
ax.barh(short_titles, csv_only, color="#4caf50", label="CSV files only (est.)")

ax.set_xlabel("Size (MB)")
ax.set_title("Storage per pipeline-ready dataset")
ax.legend()
plt.tight_layout()
plt.savefig("../ncfindia_storage.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"\nTotal MB (all files):       {ready['total_mb'].sum():.1f}")
print(f"Total MB (CSV estimate only): {csv_only.sum():.1f}")

## 8. Summary & feasibility assessment

Print a structured summary suitable for copy-pasting into the design doc.

In [ ]:
ready = datasets[datasets["pipeline_ready"]]
not_ready = datasets[~datasets["pipeline_ready"]]

print("=" * 60)
print("NCF India Zenodo — Pipeline Feasibility Summary")
print("=" * 60)
print(f"Total records in community:        {len(df)}")
print(f"Dataset records:                   {len(datasets)}")
print(f"  Pipeline-ready (CSV+codebook):   {len(ready)}")
print(f"  Not ready:                       {len(not_ready)}")
print(f"    — has CSV but no codebook:     {(datasets['has_csv'] & ~datasets['has_codebook']).sum()}")
print(f"    — has codebook but no CSV:     {(~datasets['has_csv'] & datasets['has_codebook']).sum()}")
print(f"    — neither:                     {(~datasets['has_csv'] & ~datasets['has_codebook']).sum()}")
print()
print(f"Unique researchers (dataset authors): {len(researcher_counts)}")
print()
print(f"Storage (pipeline-ready, all files):  {ready['total_mb'].sum():.1f} MB")
print(f"Storage (CSV files only, estimated):  {(ready.apply(lambda r: r['total_mb']*r['n_csv']/max(r['n_files'],1), axis=1)).sum():.1f} MB")
print()
print("Not-ready records (why):")
for _, row in not_ready[["id", "title", "has_csv", "has_codebook"]].iterrows():
    reason = []
    if not row["has_csv"]: reason.append("no CSV")
    if not row["has_codebook"]: reason.append("no codebook")
    print(f"  [{row['id']}] {row['title'][:55]}  → {', '.join(reason)}")